# Mistral 7B Playground
Load the model once (cell 1), then change the prompt in cell 2 and re-run to generate.

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "/home/azureuser/models/mistral-7b-instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="cuda:0",
)
print(f"Model loaded on {model.device}")


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded on cuda:0


In [ ]:
# === CHANGE THIS ===
prompt = "Write a Python function that checks if a number is prime."

# Sampling config
temperature = 0.7
top_p = 0.95
max_tokens = 512
# ===================

messages = [{"role": "user",
             "content": f"{prompt}\n\nReturn only code. No markdown. No explanation."}]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt").to(model.device)
attention_mask = torch.ones_like(inputs)

with torch.no_grad():
    output_ids = model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the generated tokens (skip the prompt)
generated = tokenizer.decode(
    output_ids[0][inputs.shape[1]:], skip_special_tokens=True)
print(generated)


```python
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
```
